In [1]:
import camelot
import pandas as pd
import numpy as np
import geopandas as gpd

In [2]:
votes_2022_df=pd.read_csv('../data/voter_numbers.csv')
votes_2022_df.head()

,county_id,NAME10,dem_votes,rep_votes,indep_votes,og_district
0,0,Adair,973,2166,NaN,3
1,1,Adams,510,1126,NaN,3
2,2,Allamakee,1932,3820,NaN,2
3,3,Appanoose,1412,3249,NaN,3
4,4,Audubon,633,1639,51.0,4


In [ ]:
#prior attempt at scraping https://sos.iowa.gov/elections/results/precinctvotetotals2024general.html that didn't work

# from bs4 import BeautifulSoup
# from requests import get
# from selenium import webdriver

# req = get('https://sos.iowa.gov/elections/results/precinctvotetotals2022general.html')

# page = get("https://www.kompas.com/ramadhan/jadwal-imsakiyah/kota-pangkal-pinang")
# soup = BeautifulSoup(page.text,"lxml")
# options = soup.find("select",{"name":"state"}).findAll("option")
# daerah = []

# for i in options:
#   name = i.text
#   link = i["value"];daerah.append({
#       "province": name,
#       "link": link
#   })
# soup = BeautifulSoup(str(req.content), 'lxml')
# # files = soup.find_all('a', id='cbo')

# # option = soup.find("selected",{"name":"try"}).findAll("option")
# # option_ = soup.find("select", {"style": "font-size:16px"}).findAll("option")
# # print(option)
# # print(option_)

# # for file in files:
# #     print(file['href'])


# # browser = webdriver.Chrome()
# url = ('https://sos.iowa.gov/elections/results/precinctvotetotals2022general.html')
# # browser.get(url)
# req=get(url)
# # html_source = browser.page_source
# # browser.quit()
# soup = BeautifulSoup(req.text, 'html.parser')
# for name_list in soup.find_all(type ='option'):
#     print(name_list.text)

In [3]:
raw_data=camelot.read_pdf('../data/canvsummary.pdf', pages='2-14')
raw_data

<TableList n=13>

In [ ]:
raw_data[12].df.tail(-1).info()
raw_data[0].df.tail(-1)


In [14]:
voter_data_2024=pd.concat([raw_data[i].df.tail(-1) for i in range(13)],ignore_index=True)
voter_data_2024.tail(5)


,0,1,2,3,4,5,6,7,8,9,10,11,12
296,,Total,"1,508","2,715",13,0,3,1,38,9,21,5,"4,313"
297,Wright,Election \nDay,"1,045","2,505",7,1,5,3,47,11,20,2,"3,646"
298,,Absentee,725,"1,348",2,0,0,0,18,6,23,4,"2,126"
299,,Total,"1,770","3,853",9,1,5,3,65,17,43,6,"5,772"
300,TOTAL,Election,"358,827","608,357","5,207",292,912,239,"9,025","4,317","4,477",466,"992,119"


In [15]:
cols=[4,5, 6, 7, 8, 9, 10, 11,12]
voter_data_2024.drop(voter_data_2024.columns[cols], axis=1, inplace=True)
voter_data_2024

,0,1,2,3
0,Adair,Election \nDay,558,"1,915"
1,,Absentee,528,"1,001"
2,,Total,"1,086","2,916"
3,Adams,Election \nDay,266,878
4,,Absentee,310,639
...,...,...,...,...
296,,Total,"1,508","2,715"
297,Wright,Election \nDay,"1,045","2,505"
298,,Absentee,725,"1,348"
299,,Total,"1,770","3,853"


In [18]:
votes_2024_df=voter_data_2024.loc[voter_data_2024[1]=='Total'][[2, 3]].copy()
votes_2024_df=votes_2024_df.reset_index(drop=True)

In [19]:
votes_2024_df['county_id']=votes_2022_df['county_id']
votes_2024_df['NAME10']=votes_2022_df['NAME10']
votes_2024_df['og_district']=votes_2022_df['og_district']
votes_2024_df

,2,3,county_id,NAME10,og_district
0,"1,086","2,916",0,Adair,3
1,576,"1,517",1,Adams,3
2,"2,350","4,857",2,Allamakee,2
3,"1,686","4,704",3,Appanoose,3
4,970,"2,214",4,Audubon,4
...,...,...,...,...,...
94,"1,909","3,636",94,Winnebago,4
95,"5,321","6,427",95,Winneshiek,2
96,"16,145","25,969",96,Woodbury,4
97,"1,508","2,715",97,Worth,2


In [25]:
votes_2024_df = votes_2024_df.iloc[:, [2,3,0,1,4]]

In [26]:
votes_2024_df.columns=['county_id','NAME10', 'dem_votes', 'rep_votes', 'og_district']
votes_2024_df

,county_id,NAME10,dem_votes,rep_votes,og_district
0,0,Adair,"1,086","2,916",3
1,1,Adams,576,"1,517",3
2,2,Allamakee,"2,350","4,857",2
3,3,Appanoose,"1,686","4,704",3
4,4,Audubon,970,"2,214",4
...,...,...,...,...,...
94,94,Winnebago,"1,909","3,636",4
95,95,Winneshiek,"5,321","6,427",2
96,96,Woodbury,"16,145","25,969",4
97,97,Worth,"1,508","2,715",2


In [27]:
votes_2024_df=votes_2024_df.replace(',','',regex=True)

In [30]:
votes_2024_df['dem_votes']=votes_2024_df['dem_votes'].astype(int)
votes_2024_df['rep_votes']=votes_2024_df['rep_votes'].astype(int)
votes_2024_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   county_id    99 non-null     int64 
 1   NAME10       99 non-null     object
 2   dem_votes    99 non-null     int64 
 3   rep_votes    99 non-null     int64 
 4   og_district  99 non-null     int64 
dtypes: int64(4), object(1)
memory usage: 4.0+ KB


In [34]:
votes_2024_df.to_csv('../data/voter_numbers_2024.csv', index=False,header=True)

In [35]:
df=pd.read_csv('../data/voter_numbers_2024.csv')
df

,county_id,NAME10,dem_votes,rep_votes,og_district
0,0,Adair,1086,2916,3
1,1,Adams,576,1517,3
2,2,Allamakee,2350,4857,2
3,3,Appanoose,1686,4704,3
4,4,Audubon,970,2214,4
...,...,...,...,...,...
94,94,Winnebago,1909,3636,4
95,95,Winneshiek,5321,6427,2
96,96,Woodbury,16145,25969,4
97,97,Worth,1508,2715,2
